# 4 - FWI is just training a model (PyTorch nn.Module style)

Full-waveform inversion looks exotic, but it is the **same loop as neural-network training**. The only unusual part is that the 'network' is a wave-equation simulator instead of a stack of linear layers - everything else is standard PyTorch.

| Neural-network training | This FWI |
|---|---|
| model (`nn.Module`) | the differentiable wave-equation solver |
| weights (`nn.Parameter`) | the material model `alpha2` (squared wave speed) |
| forward pass | solve the wave equation -> predicted seismograms |
| loss | L2 waveform misfit vs the recorded data |
| `loss.backward()` | the adjoint-state gradient, computed automatically |
| `optimizer.step()` | update the model (here L-BFGS) |
| training data | seismograms recorded through the true (cracked) plate |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seidlr/acoustic-fwi-ndt-pytorch/blob/main/notebooks/04_fwi_as_nn_training.ipynb)

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    OWNER = "seidlr"  # change to your account if you forked
    !git clone -q https://github.com/{OWNER}/acoustic-fwi-ndt-pytorch.git
    %cd acoustic-fwi-ndt-pytorch
    sys.path.insert(0, os.path.abspath("src"))  # make `import fwi` importable now

import torch
from torch import nn
import matplotlib.pyplot as plt
from fwi.config import resolve_device, resolve_dtype

device = resolve_device()
dtype = resolve_dtype(device)
print("device:", device, "| dtype:", dtype)

## The 'network': a differentiable wave solver

This is the forward model. It time-steps the 2D acoustic wave equation and returns the receiver traces. It is pure PyTorch, so autograd can differentiate the misfit straight through every timestep - that is the adjoint-state gradient, for free. Here is its actual source:

In [ ]:
import inspect
from fwi.forward import forward
print(inspect.getsource(forward))

## Wrap it as an `nn.Module`

The learnable weight is a dimensionless model `m = alpha2 / alpha2_background` (so it is ~1, like well-scaled NN weights). `forward()` turns `m` back into a physical `alpha2` and runs the solver. The `clamp` keeps every step CFL-stable and the `mask` freezes the fixed boundary (ghost) cells - both are differentiable, so the training loop below stays a plain PyTorch loop.

In [ ]:
class AcousticFWI(nn.Module):
    """The wave solver as a trainable model: weights = the material model."""

    def __init__(self, m_init, src_sig, src_i, src_j, rec_i, rec_j,
                 cfg, active_mask, m_max, alpha2_bg):
        super().__init__()
        self.m = nn.Parameter(m_init.clone())          # what we invert for
        self.register_buffer('src_sig', src_sig)
        self.register_buffer('src_i', src_i)
        self.register_buffer('src_j', src_j)
        self.register_buffer('rec_i', rec_i)
        self.register_buffer('rec_j', rec_j)
        self.register_buffer('mask', active_mask.to(m_init.dtype))
        self.cfg, self.m_max, self.alpha2_bg = cfg, m_max, alpha2_bg

    def forward(self):
        # dimensionless m -> CFL-safe, boundary-masked physical alpha2
        alpha2 = self.m.clamp(0.0, self.m_max) * self.mask * self.alpha2_bg
        return forward(alpha2, self.src_sig, self.src_i, self.src_j,
                       self.rec_i, self.rec_j, self.cfg).traces

## Load the data, define the loss, build the model

`build_problem('crack')` gives the start model (homogeneous aluminium), the true model (one crack), and the **observed** seismograms recorded through the true model - our training data. The loss is the L2 waveform misfit, normalised by its starting value `J0` so it begins at 1.0.

In [ ]:
from fwi.problems import build_problem
from fwi.config import SPEED_ALUMINUM

prob = build_problem('crack', device=device, dtype=dtype)
observed = prob.observed
alpha2_bg = SPEED_ALUMINUM ** 2
m_init = (prob.start_alpha2 / alpha2_bg)
# CFL-safe upper bound on m (same guard the library uses)
c_limit = prob.cfg.cfl * min(prob.cfg.dx_m, prob.cfg.dy_m) / prob.cfg.dt
m_max = 0.9 * c_limit ** 2 / alpha2_bg

model = AcousticFWI(m_init, prob.src_sig, prob.src_i, prob.src_j,
                    prob.rec_i, prob.rec_j, prob.cfg, prob.active_mask,
                    m_max, alpha2_bg).to(device)

with torch.no_grad():
    J0 = float(0.5 * ((model() - observed) ** 2).sum() * prob.cfg.dt)

def waveform_loss(pred, target):
    return 0.5 * ((pred - target) ** 2).sum() * prob.cfg.dt / J0

print('trainable parameters:', sum(p.numel() for p in model.parameters()))
print(f'start loss (J/J0): {waveform_loss(model(), observed).item():.4f}')

## Train it - the classic PyTorch loop

`torch.optim.LBFGS` needs a `closure` that re-evaluates the model and calls `loss.backward()` - identical to any second-order PyTorch training. Each `opt.step(closure)` does one L-BFGS update; we just log the loss as it goes.

In [ ]:
N_EPOCHS = 25
opt = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=1,
                        line_search_fn='strong_wolfe')
losses = []

def closure():
    opt.zero_grad()
    loss = waveform_loss(model(), observed)
    loss.backward()
    return loss

for epoch in range(N_EPOCHS):
    loss = opt.step(closure)
    losses.append(float(loss))
    print(f'epoch {epoch + 1:02d}/{N_EPOCHS} | loss (J/J0) = {float(loss):.4e}')

print(f'\nloss reduced {losses[0] / losses[-1]:.1f}x  ({losses[0]:.3e} -> {losses[-1]:.3e})')

## What the model learned

The recovered weight update (recovered `alpha2` minus the homogeneous start) should localise the crack, matching the true perturbation.

In [ ]:
with torch.no_grad():
    alpha2_hat = (model.m.clamp(0.0, m_max) * model.mask * alpha2_bg)
recovered = (alpha2_hat - prob.start_alpha2).detach().cpu().numpy()
truth = (prob.true_alpha2 - prob.start_alpha2).detach().cpu().numpy()

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].semilogy(range(1, len(losses) + 1), losses, 'o-')
ax[0].set(title='training loss', xlabel='epoch', ylabel='J / J0'); ax[0].grid(True, alpha=0.3)
vmax = abs(truth).max()
ax[1].imshow(recovered, cmap='RdBu_r', vmin=-vmax, vmax=vmax); ax[1].set_title('recovered crack')
ax[2].imshow(truth, cmap='RdBu_r', vmin=-vmax, vmax=vmax); ax[2].set_title('true crack')
for a in ax[1:]: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()